# Подготовка данных

In [ ]:
# Импорты

!pip install rouge
!pip install rouge_score
!pip install torch
import pandas as pd
import os
import torch
from transformers import GPT2Tokenizer
from transformers import pipeline
from rouge_score import rouge_scorer
from torch.utils.data import DataLoader
import sys
current_dir = os.getcwd()
sys.path.insert(0, current_dir)
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

In [10]:
# Сбор и подготовка данных
output_dir='data/'
from src.data_utils import *
df=load_and_clean_data('https://code.s3.yandex.net/deep-learning/tweets.txt')
train_df, val_df, test_df=split_dataset(df)
save_datasets(train_df, val_df, test_df, output_dir)

Файл успешно записан: data//train.csv
Файл успешно записан: data//val.csv
Файл успешно записан: data//test.csv
train: 1277895, val: 159737, test: 159737


In [ ]:
# DataLoader
from transformers import GPT2Tokenizer
from src.next_token_dataset import NextTokenDataset

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained("models/tokenizer/")

train_texts = pd.read_csv("data/train.csv")['text'].tolist()[:10000]
val_texts = pd.read_csv("data/val.csv")['text'].tolist()[:2000]

train_dataset = NextTokenDataset(train_texts, tokenizer, max_length=20)
val_dataset = NextTokenDataset(val_texts, tokenizer, max_length=20)

def collate_fn(batch, pad_token_id=50256):
    import torch
    from torch import nn

    input_ids = [torch.tensor(item['input_ids']) for item in batch]
    labels = [torch.tensor(item['labels']) for item in batch]

    # Паддинг до максимальной длины в батче
    input_ids = nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    labels = nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=pad_token_id)

    # Возвращаем словарь, а не кортеж!
    return {
        'input_ids': input_ids,
        'labels': labels
    }

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# Реализация сети

In [3]:
# Загрузка данных и модель
from src.lstm_train import train_model
device = 'cpu'
# Обучение
trained_model = train_model(train_loader, val_loader,vocab_size=tokenizer.vocab_size, epochs=7, device=device)

# Сохранение модели
torch.save(trained_model.state_dict(), 'models/lstm_model.pth')
print("Модель сохранена: models/lstm_model.pth")

Epoch 1/7, Loss: 5.7878
Epoch 2/7, Loss: 4.6176
Epoch 3/7, Loss: 4.5326
Epoch 4/7, Loss: 4.5008
Epoch 5/7, Loss: 4.4703
Epoch 6/7, Loss: 4.4555
Epoch 7/7, Loss: 4.4408
Модель сохранена: models/lstm_model.pth


## Оценка LSTM

In [4]:
# Оценка LSTM
from src.eval_lstm import evaluate_lstm
lstm_rouge1, lstm_rouge2 = evaluate_lstm(trained_model, val_loader, tokenizer, device=device)

Evaluating LSTM: 100%|██████████| 32/32 [07:47<00:00, 14.60s/it]


LSTM ROUGE-1: 0.3351

LSTM ROUGE-2: 0.2911


## Примеры промптов LSTM

In [8]:
# Примеры промптов — начала фраз
examples = [
    "I'm happy",
    "I was too ",
    "i feel",
    "It just seems",
    "i want to"
]
print("Оценка автодополнения LSTM:")
for prompt in examples:
    generated = trained_model.generate(tokenizer, prompt, max_length=20, device=device)
    print(f"Промпт: {prompt}")
    print(f"Дополнение LSTM: {generated}")

Оценка автодополнения LSTM:
Промпт: I'm happy
Дополнение LSTM: i'm happyus web ise i you to you dont tolday wish stupid a myselfark my
Промпт: I was too 
Дополнение LSTM: i was too  starts me am ll so u he followers i cant th stop then great didnt more
Промпт: i feel
Дополнение LSTM: i feel more beay got that muss until sad wake couldk youtubehex the gadgets anything
Промпт: It just seems
Дополнение LSTM: it just seems its sorry i have a fried much 6 friends yours 2 usesuc2003 happen but fun
Промпт: i want to
Дополнение LSTM: i want to welcome love likethis get and


# Предобученный трансформер

In [6]:
# Оценка DistilGPT-2
from src.eval_transformer_pipeline import evaluate_transformer
r1, r2 = evaluate_transformer(val_loader, tokenizer, device=device, max_examples=100)

DistilGPT-2: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]

DistilGPT-2 (на 100 примерах):
  ROUGE-1: 0.6746
  ROUGE-2: 0.6251


## Примеры промптов трансформера

In [9]:
# Сравнение результатов LSTM и DistilGPT-2
# Загружаем модель DistilGPT-2
generator_DistilGPT = pipeline("text-generation", model="distilgpt2")

# Примеры промптов — начала фраз
examples = [
    "I'm happy",
    "I was too ",
    "i feel",
    "It just seems",
    "i want to"
]
print("Оценка автодополнения DistilGPT:")
for prompt in examples:
    result = generator_DistilGPT(prompt, max_length=20, do_sample=True, top_k=50)
    generated = result[0]['generated_text']
    print(f"Промпт: {prompt}")
    print(f"Дополнение DistilGPT: {generated}")

Оценка автодополнения DistilGPT:
Промпт: I'm happy
Дополнение DistilGPT: I'm happy with how I do it. Even though I can't live with all of it, I just want to see the pain of what I've spent it on.

I'm trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. My job is to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but it's not working at all.
For me, I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good person, but I'm still trying to be a good person, but it's not working at all. I'm still trying to be a good

# Сравнение моделей

При оценке по ROUGE-метрикам LSTM показывает невысокие результаты (ROUGE-1 ~ 0,335; ROUGE-2~ 0,291) и генерирует тексты c низким качеством: использует несуществующие слова и создаёт бессвязные фразы. DistilGPT-2, напротив, достигает высоких показателей (ROUGE-1 ~ 0,675; ROUGE-2 ~ 0,625) и формирует предложения, отличающиеся логичностью, связностью и естественностью звучания, хотя при этом присутсвует повторение финальных фраз.